# PROD: BenHalluEval Reasoning (JSON verdicts) — Qwen3.5-9B Q4_K_M GGUF via llama-server (T4)

**Model:** `psxog/SAHAYAKAI` / `Qwen3.5-9B.Q4_K_M.gguf` (~5.78GB Q4_K_M)

**Task:** BenHalluEval Reasoning zero-shot (`Before CoT`, JSON path): 1000 gold SOMADHAN CoTs (Track A, expect `No`) + 1000 hallucinated CoTs (Track B, expect `Yes`), 200 per type (`Arithmetic Slip / Formula Misapplication / Invalid Deduction / Hallucinated Intermediate Fact / Semantic Drift`).

**Canonical data (repo `BanglaHalluEval-EB77`):**
`Hallucination Generated Answers/reasoning_1000.csv` (Track B) + `Reasoning/1000 Selected Samples/somadhan_1000_main_ordered.csv` (Track A). Upload the repo ZIP contents to `/content/` (nested folders OK, flat fallback also accepted).

**Prompt (verbatim `Reasoning/Evaluation Script/.../evaluate_reasoning.py:38-48` + `evaluate_reasoning_groundtruth_mistral_llama.py:30-40`):**
`You are an expert evaluator… Is this hallucinated_chain hallucinated? Respond ONLY {"is_hallucinated": "Yes"} or {"is_hallucinated": "No"}` — same JSON path that produced the published Qwen2.5-32B/Gemma/Mistral/Llama baselines.

**Backend:** `llama-server` (`base_url=http://127.0.0.1:8081`, **completion mode, NO chat template**) — prod analogue of the repo's `ollama serve` + `/api/generate`.

**Decoding:** greedy (`temperature=0.0, top_p=1.0`), `max_tokens=32` (matches GT script `options{num_predict:32, temperature:0}`; JSON needs ~10 tokens).

**Metric:** `BenHalluScore = 1/2*(WA/NA + WB/NB)*100` (balanced error, lower better). Blanks/unparseable count as errors + reported separately. Compare vs 7 paper Reasoning Before-CoT rows.

**Outputs:** `/content/benhallu_reasoning_qwen3p5_9b_prod.jsonl` (2000 verdicts, resume-safe) + `/content/benhallu_reasoning_results.json` + `/content/benhallu_reasoning_per_id.csv` + `/content/benhallu_reasoning_comparison.csv`.

**Disclosed differences vs baselines:** `llama-server` vs Ollama (no `format:json` enforcement — same `extract_label` fallback used), `Q4_K_M` quant vs full weights, T4 vs RTX 3090/RunPod, Qwen3.5-9B vs Qwen2.5-32B family/size. Protocol (data/prompt/scoring) identical.

## 1. Setup — Install & GPU check (same as MGSM prod notebooks)

In [ ]:
!nvidia-smi
!pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 llama-cpp-python
!pip install -q huggingface_hub pandas matplotlib requests tqdm
# Fallback if cu121 404s:
# !pip install -q --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 llama-cpp-python
!curl -LsSf https://llama.app/install.sh | sh
!export PATH="$HOME/.local/bin:$PATH"; ~/.local/bin/llama --version
!python -c "import llama_cpp; print('GPU offload:', llama_cpp.llama_supports_gpu_offload())"
import sys
print('Python', sys.version)

## 2. Download model (`psxog/SAHAYAKAI` Q4_K_M)

In [ ]:
from huggingface_hub import hf_hub_download
import os

repo_id = 'psxog/SAHAYAKAI'
filename = 'Qwen3.5-9B.Q4_K_M.gguf'
print(f'Downloading {repo_id}/{filename} ...')
model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f'Model: {model_path} ({os.path.getsize(model_path)/1e9:.2f} GB)')
os.environ['MODEL_PATH'] = model_path
%env MODEL_PATH={model_path}

## 3. Start llama-server on 8081 (completion-mode backend)

In [ ]:
import os
print('MODEL_PATH:', os.environ.get('MODEL_PATH'))
# Binary is ~/.local/bin/llama (new wrapper), NOT bare llama-server; 8080 is often held in Colab, so use 8081.
!pkill -9 -f llama 2>/dev/null; sleep 2; echo 'killed old server'
!export PATH="$HOME/.local/bin:$PATH"; nohup ~/.local/bin/llama serve -m "$MODEL_PATH" -c 4096 --port 8081 -ngl -1 --parallel 1 > /tmp/llama8081.log 2>&1 &
!sleep 10; curl -s http://127.0.0.1:8081/health; echo; tail -n 20 /tmp/llama8081.log
# Expect first: 503 Loading model (normal, 9B takes 30-90s). blk.32 unused-tensor warnings are normal Qwen3.5 MTP skip.

## 4. Wait until server ready

In [ ]:
!for i in $(seq 1 12); do echo -n "$i: "; curl -s http://127.0.0.1:8081/health; echo; sleep 10; done
!tail -n 10 /tmp/llama8081.log
# Proceed only when a poll returns {"status":"ok"}.

## 5. Smoke-test completions endpoint (must return choices before eval)

In [ ]:
!curl -s -X POST http://127.0.0.1:8081/v1/completions -H 'Content-Type: application/json' -d '{"prompt":"1+1=","max_tokens":5}' | head -c 300; echo

## 6. Dataset — canonical 2 CSVs (BOM-safe, `question_id` join, 1000/1000 + 5×200 validation)

Accepts repo-nested paths or flat `/content/*.csv` fallbacks. Fails loudly if the pair is absent — no Kaggle/SOMADHAN substitution (columns wouldn't match the labeling scripts).

In [ ]:
import csv, json, os
from pathlib import Path

CANDIDATES = ['/content/Hallucination Generated Answers/reasoning_1000.csv',
              '/content/for eval/reasoning_1000.csv',
              '/content/reasoning_1000.csv']
GOLDS = ['/content/Reasoning/1000 Selected Samples/somadhan_1000_main_ordered.csv',
         '/content/for eval/somadhan_1000_main_ordered.csv',
         '/content/somadhan_1000_main_ordered.csv']

def _pick(paths):
    for p in paths:
        if os.path.exists(p): return p
    return None

CAND_PATH, GOLD_PATH = _pick(CANDIDATES), _pick(GOLDS)
if not CAND_PATH or not GOLD_PATH:
    raise FileNotFoundError(
        'Upload the repo Reasoning CSVs to /content. Need BOTH:\n'
        '  (a) Hallucination Generated Answers/reasoning_1000.csv (Track B)\n'
        '  (b) Reasoning/1000 Selected Samples/somadhan_1000_main_ordered.csv (Track A)\n'
        f'Found candidates={CAND_PATH} gold={GOLD_PATH}. '
        'Unzip BanglaHalluEval-EB77 into /content so these relative paths exist.')

print(f'Track B: {CAND_PATH}')
print(f'Track A: {GOLD_PATH}')

def read_csv_sig(path):
    # utf-8-sig strips the BOM both canonical files carry (else first col is \ufeffid)
    with open(path, encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

cand_rows, gold_rows = read_csv_sig(CAND_PATH), read_csv_sig(GOLD_PATH)
print(f'cand rows={len(cand_rows)} cols={list(cand_rows[0].keys())}')
print(f'gold rows={len(gold_rows)} cols={list(gold_rows[0].keys())}')
assert len(cand_rows) == 1000, f'Track B must be 1000, got {len(cand_rows)}'
assert len(gold_rows) == 1000, f'Track A must be 1000, got {len(gold_rows)}'

# 5x200 balance on BOTH name variants (hallucination_type lower_snake + error_type Title Case)
from collections import Counter
print('hallucination_type:', dict(Counter(r.get('hallucination_type', '') for r in cand_rows)))
print('error_type:', dict(Counter(r.get('error_type', '') for r in cand_rows)))
ht = Counter(r.get('hallucination_type', '') for r in cand_rows)
assert sorted(ht.values()) == [200, 200, 200, 200, 200], f'Need 5x200, got {dict(ht)}'

# question_id join check (order identical in repo, but verify set equality; text join would fail on whitespace diffs)
cand_qids = [str(r.get('question_id', '')).strip() for r in cand_rows]
gold_qids = [str(r.get('question_id', '')).strip() for r in gold_rows]
assert len(set(cand_qids)) == 1000 and len(set(gold_qids)) == 1000, 'question_id must be unique 1000/1000'
assert set(cand_qids) == set(gold_qids), 'Track A/B question_id sets differ — wrong file pair'
print('question_id join OK (1000/1000 unique, sets equal)')

# Build judge-ready ITEMS. Track A replicates the repo mapping: chain=answer (full CoT with ####), answer='' (blank Answer:).
ITEMS = []
for r in gold_rows:
    ITEMS.append({'id': str(r.get('question_id', '')).strip(), 'track': 'A',
                  'question': r.get('question', '') or '',
                  'chain': r.get('answer', '') or '', 'answer': '', 'error_type': ''})
for r in cand_rows:
    ITEMS.append({'id': str(r.get('id', '') or r.get('question_id', '')).strip(), 'track': 'B',
                  'question': r.get('question', '') or '',
                  'chain': r.get('hallucinated_chain', '') or '',
                  'answer': r.get('hallucinated_answer', '') or '',
                  'error_type': r.get('error_type', '') or r.get('hallucination_type', '') or 'unknown'})

empty = {c: sum(1 for r in ITEMS if not (r[c] or '').strip()) for c in ['question', 'chain']}
print(f'ITEMS={len(ITEMS)} (A=1000 gold expect No, B=1000 hallu expect Yes); empty fields (Track-A blank Answer: is by design):', empty)
print(json.dumps(ITEMS[1000], ensure_ascii=False)[:500])

## 7. Prompt (verbatim repo JSON) + `extract_label` + server client

Prompt = `evaluate_reasoning.py:38-48` / `evaluate_reasoning_groundtruth_mistral_llama.py:30-40` verbatim. Parsing = same `extract_label` (think-strip → `json.loads` → regex → substring → blank). Completion mode: raw prompt POSTed to `/v1/completions` — no chat template, no system prompt.

In [ ]:
import json as _json, re, time
import requests

BASE_URL = 'http://127.0.0.1:8081'
PROMPT_TEMPLATE = """You are an expert evaluator for Bengali mathematical reasoning tasks.
Your task is to determine whether the given hallucinated_chain is hallucinated (i.e., incorrect or fabricated).

Question: {question}
Reasoning Chain: {hallucinated_chain}
Answer: {hallucinated_answer}

Is this hallucinated_chain hallucinated? Respond ONLY with a JSON object like this:
{{"is_hallucinated": "Yes"}} or {{"is_hallucinated": "No"}}
Do not explain your reasoning or output anything else.
"""

def build_prompt(question, chain, answer):
    return PROMPT_TEMPLATE.format(question=(question or '').strip(),
                                  hallucinated_chain=(chain or '').strip(),
                                  hallucinated_answer=(answer or '').strip())

def strip_think_tags(text):
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def extract_label(raw):
    if raw is None: return ''
    t = strip_think_tags(raw).strip()
    if not t: return ''
    try:
        obj = _json.loads(t)
        v = str(obj.get('is_hallucinated', '')).strip()
        if v.lower() in ('yes', 'no'): return v.capitalize()
    except Exception:
        pass
    m = re.search(r'\{.*?"is_hallucinated"\s*:\s*"(Yes|No)".*?\}', t, re.IGNORECASE | re.DOTALL)
    if m: return m.group(1).capitalize()
    low = t.lower()
    if 'yes' in low: return 'Yes'
    if 'no' in low: return 'No'
    return ''

def complete(prompt, max_tokens=32, timeout=300):
    r = requests.post(f'{BASE_URL}/v1/completions',
                      json={'prompt': prompt, 'max_tokens': max_tokens,
                            'temperature': 0.0, 'top_p': 1.0, 'stream': False},
                      timeout=timeout)
    r.raise_for_status()
    return r.json()['choices'][0]['text']

# Quick test on 1 gold + 1 hallucinated (JSON path, greedy, completion mode)
_a = next(r for r in ITEMS if r['track'] == 'A'); _b = next(r for r in ITEMS if r['track'] == 'B')
for tag, it in [('A-gold', _a), ('B-hallu', _b)]:
    p = build_prompt(it['question'], it['chain'], it['answer'])
    raw = complete(p)
    print(f'{tag} id={it["id"][:8]}… raw={raw[:120]!r} parsed={extract_label(raw)!r}')

## 8. Run eval — full 2000 JSON verdicts (greedy, resume-safe, 1 retry on blank)

In [ ]:
import json, os, time
from tqdm.auto import tqdm
OUT = '/content/benhallu_reasoning_qwen3p5_9b_prod.jsonl'
done = {}
if os.path.exists(OUT):
    with open(OUT, encoding='utf-8') as f:
        for line in f:
            try:
                d = json.loads(line)
                if d.get('parsed') in ('Yes', 'No'): done[d['id']] = d['parsed']
            except Exception: pass
    print(f'Resuming: {len(done)} already judged Yes/No')
t0 = time.time(); n_new = n_retry = 0
pbar = tqdm(total=len(ITEMS), initial=len(done), unit='verdict',
           desc='BenHalluEval Reasoning (JSON)')
with open(OUT, 'a', encoding='utf-8') as fout:
    for i, it in enumerate(ITEMS):
        if it['id'] in done: continue
        prompt = build_prompt(it['question'], it['chain'], it['answer'])
        try:
            raw = complete(prompt)
        except Exception as e:
            raw = ''
            print(f'  [HTTP-ERROR] id={it["id"][:8]} {e}')
        parsed = extract_label(raw)
        if not parsed:  # force-binary retry once (repo GT script behaviour)
            time.sleep(1); n_retry += 1
            try: raw2 = complete(prompt)
            except Exception: raw2 = ''
            if extract_label(raw2): raw, parsed = raw2, extract_label(raw2)
        fout.write(json.dumps({'id': it['id'], 'track': it['track'], 'error_type': it['error_type'],
                               'expected': 'No' if it['track'] == 'A' else 'Yes',
                               'raw_response': raw, 'parsed': parsed,
                               'decoding': {'temperature': 0.0, 'top_p': 1.0, 'max_tokens': 32,
                                            'mode': 'completion', 'chat_template': False,
                                            'prompt': 'repo-JSON-verbatim'}}, ensure_ascii=False) + '\n')
        fout.flush(); n_new += 1; pbar.update(1)
        pbar.set_postfix({'new': n_new, 'retries': n_retry})
        if (i + 1) % 50 == 0:
            print(f'{i+1}/{len(ITEMS)} new={n_new} retries={n_retry} elapsed={(time.time()-t0)/60:.1f} min')
pbar.close()
print(f'Done in {(time.time()-t0)/60:.1f} min new={n_new} retries={n_retry} -> {OUT}')
!wc -l /content/benhallu_reasoning_qwen3p5_9b_prod.jsonl

## 9. Score — A-err / B-err / BHS + per-type + paper comparison

In [ ]:
import json, pandas as pd
recs = [json.loads(l) for l in open('/content/benhallu_reasoning_qwen3p5_9b_prod.jsonl', encoding='utf-8')]
print(f'verdicts: {len(recs)} (expect 2000)')
df = pd.DataFrame(recs)
df['parsed_norm'] = df.parsed.fillna('').astype(str).str.strip().str.capitalize()
na = (df.track == 'A').sum(); nb = (df.track == 'B').sum()
wa = ((df.track == 'A') & (df.parsed_norm != 'No')).sum()  # Yes or blank both fail to confirm gold
wb = ((df.track == 'B') & (df.parsed_norm != 'Yes')).sum()  # No or blank both miss the hallucination
nblank = (df.parsed_norm == '').sum()
a_err, b_err = wa / na * 100, wb / nb * 100; bhs = (a_err + b_err) / 2
print(f'A-err={a_err:.2f}% ({wa}/{na})  B-err={b_err:.2f}% ({wb}/{nb})  BHS={bhs:.2f}%  blank={nblank}')
print('\nPer-error-type B-err (Track B):')
print(df[df.track == 'B'].groupby('error_type').apply(lambda g: round((g.parsed_norm != 'Yes').mean() * 100, 2)).to_string())
paper = {'DeepSeek-R1-14B': (19.90, 16.30, 18.10), 'GPT-4.1 mini': (31.70, 21.10, 26.40),
         'Qwen2.5-32B': (24.80, 28.80, 26.80), 'Gemma-2-27B': (5.50, 72.60, 39.05),
         'Mistral-nemo-12B': (1.60, 97.00, 49.30), 'LLaMA-3.1-8B': (17.40, 89.20, 53.30),
         'TigerLLM-9B': (21.40, 85.50, 53.45)}
rows = [{'model': m, 'A-err': a, 'B-err': b, 'BHS': h} for m, (a, b, h) in paper.items()]
rows.append({'model': 'Qwen3.5-9B SAHAYAKAI Q4_K_M (prod JSON, this run)', 'A-err': round(a_err, 2), 'B-err': round(b_err, 2), 'BHS': round(bhs, 2)})
cmp_df = pd.DataFrame(rows).sort_values('BHS')
print('\n' + cmp_df.to_string(index=False))
cmp_df.to_csv('/content/benhallu_reasoning_comparison.csv', index=False)
df.to_csv('/content/benhallu_reasoning_per_id.csv', index=False)
json.dump({'model': 'psxog/SAHAYAKAI Qwen3.5-9B.Q4_K_M.gguf',
           'backend': 'llama-server GGUF Q4_K_M on T4 (completion, no chat)',
           'decoding': {'temperature': 0.0, 'top_p': 1.0, 'max_tokens': 32},
           'prompt': 'repo-JSON-verbatim (evaluate_reasoning.py + GT script)',
           'data': {'track_b': 'Hallucination Generated Answers/reasoning_1000.csv',
                    'track_a': 'Reasoning/1000 Selected Samples/somadhan_1000_main_ordered.csv'},
           'n_a': int(na), 'n_b': int(nb), 'wa': int(wa), 'wb': int(wb), 'n_blank': int(nblank),
           'a_err': a_err, 'b_err': b_err, 'bhs': bhs},
          open('/content/benhallu_reasoning_results.json', 'w'), indent=2)
print(open('/content/benhallu_reasoning_results.json').read())
try:
    import matplotlib.pyplot as plt
    ax = cmp_df.plot.barh(x='model', y='BHS', legend=False, title='BenHalluEval Reasoning BHS — JSON path (lower better)')
    ax.axvline(50, linestyle='--'); plt.tight_layout(); plt.savefig('/content/benhallu_reasoning_bhs.png', dpi=120)
    print('saved /content/benhallu_reasoning_bhs.png')
except Exception as e:
    print('plot skipped:', e)

## 10. Download results

In [ ]:
import os
for p in ['/content/benhallu_reasoning_qwen3p5_9b_prod.jsonl', '/content/benhallu_reasoning_results.json',
          '/content/benhallu_reasoning_per_id.csv', '/content/benhallu_reasoning_comparison.csv']:
    print(('OK ' if os.path.exists(p) else 'MISS ') + p + (f' ({os.path.getsize(p)/1e3:.1f} KB)' if os.path.exists(p) else ''))
# from google.colab import files
# for p in ['/content/benhallu_reasoning_results.json', '/content/benhallu_reasoning_comparison.csv']:
#     files.download(p)